In [16]:
%load_ext rpy2.ipython

In [2]:
import json
import pandas as pd
from pathlib import Path

In [28]:
NycAvi_absrel = list(Path('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/aBSREL/NycAvi').glob('*.json'))
IaIo_absrel = list(Path('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/aBSREL/IaIo').glob('*.json'))

In [26]:
def parse(path_lst, specie):
    results = {"file": [], "pvalue": []}
    for json_file in path_lst:
        try:
            js = json.load(open(json_file, "r"))
            pvalue = js["branch attributes"]["0"][specie]["Corrected P-value"]
            if pvalue <= 0.05:
                results["file"].append(json_file.stem)
                results["pvalue"].append(pvalue)
        except json.JSONDecodeError:
            print(json_file)

    return pd.DataFrame(results)

In [30]:
NycAvi_df = parse(NycAvi_absrel, 'NycAvi')
IaIo_df = parse(IaIo_absrel, 'IaIo')

In [31]:
%R -i NycAvi_df
%R -i IaIo_df

In [34]:
%%R
NycAvi_df$padj = p.adjust(NycAvi_df$pvalue, method = 'fdr')
IaIo_df$padj = p.adjust(IaIo_df$pvalue, method = 'fdr')

In [35]:
NycAvi_df = %R NycAvi_df
IaIo_df = %R IaIo_df

In [38]:
NycAvi_df = NycAvi_df[NycAvi_df.padj < 0.05]
IaIo_df = IaIo_df[IaIo_df.padj < 0.05]

In [39]:
NycAvi_df.to_csv(path_or_buf='/home/panda2bat/Avivorous_bat/output/14_evolution-selective/aBSREL/NycAvi.sig.psg.tsv', sep='\t', index=False)
IaIo_df.to_csv(path_or_buf='/home/panda2bat/Avivorous_bat/output/14_evolution-selective/aBSREL/IaIo.sig.psg.tsv', sep='\t', index = False)